# Fine-tune a Sentinel moderation judge (QLoRA)

Runs on a free Colab **T4 GPU**. It builds a clean dataset, QLoRA-tunes an
LLM into a 12-category moderation judge, and shows how to plug it back into
the chat app.

> Runtime → Change runtime type → **T4 GPU** before you start.

In [ ]:
# 1) Get the code + training deps
!git clone https://github.com/stevenkayitaresteven/portifolio.git
%cd portifolio
!pip install -q -e ".[data,train]" peft trl bitsandbytes accelerate

In [ ]:
# 2) Build a dataset. The demo uses a synthetic sample; for a real run, point
#    a build config at public datasets (Jigsaw, Civil Comments, Aegis 2.0...).
!python -m safety.data demo --out runs/corpus
!head -n 3 runs/corpus/train.jsonl

In [ ]:
# 3) Validate data + config on CPU first (no model download)
!python -m safety.train.lora.finetune_llm \
    --config safety/train/lora/configs/llama31_8b_moderator.json \
    --data runs/corpus --dry-run

In [ ]:
# 4) Train (GPU). Swap the config to fine-tune Mistral 7B / Gemma 3 / Qwen, etc.
#    Gated models (Llama) need: from huggingface_hub import login; login('hf_...')
!python -m safety.train.lora.finetune_llm \
    --config safety/train/lora/configs/llama31_8b_moderator.json \
    --data runs/corpus --out runs/llama31-moderator

In [ ]:
# 5) Use it as the live judge in the chat app
import os
os.environ['SAFETY_MM_USE_LLM_JUDGE'] = '1'
os.environ['SAFETY_MM_LLM_JUDGE_MODEL'] = 'runs/llama31-moderator'
from safety.multimodal import MultimodalModerator
mod = MultimodalModerator()
print(mod.moderate_text('you are absolutely worthless').to_dict())